# AI Bias & Fairness Detector

This notebook demonstrates the end-to-end pipeline of the project.

In [ ]:
from src.preprocess import load_data, preprocess_data, get_train_test_splits
from src.model import train_baseline_models, evaluate_model
from src.bias_metrics import calculate_fairness_metrics
from src.mitigation import mitigation_threshold_optimizer, mitigation_exponentiated_gradient
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

## 1. Load and Preprocess Data

In [ ]:
df = load_data()
X, y, sf = preprocess_data(df)
X_tr, X_te, y_tr, y_te, sf_tr, sf_te = get_train_test_splits(X, y, sf)
print(f"Train shape: {X_tr.shape}, Test shape: {X_te.shape}")

## 2. Train Baseline Model

In [ ]:
lr_baseline, _ = train_baseline_models(X_tr, y_tr)
metrics_lr, group_acc_lr = evaluate_model(lr_baseline, X_te, y_te, sf_te)
print("Baseline Logistic Regression Accuracy:", metrics_lr['accuracy'])

## 3. Detect Bias

In [ ]:
y_pred_baseline = lr_baseline.predict(X_te)
fairness_baseline = calculate_fairness_metrics(y_te, y_pred_baseline, sf_te)
print("Baseline Fairness Metrics:")
for k, v in fairness_baseline.items():
    print(f"  {k}: {v}")

## 4. Mitigate Bias (Post-processing: Threshold Optimizer)

In [ ]:
to_mitigator = mitigation_threshold_optimizer(lr_baseline, X_tr, y_tr, sf_tr)
y_pred_to = to_mitigator.predict(X_te, sensitive_features=sf_te['sex'])
fairness_to = calculate_fairness_metrics(y_te, y_pred_to, sf_te)
print("Metrics after Post-processing (Threshold Optimizer):")
for k, v in fairness_to.items():
    print(f"  {k}: {v}")